# Titanic – Experiments: XGBoost, group features and probability blending
This notebook continues from `Titanic_Random_Forest_Classification_v2.ipynb`, whose family-group rule scored **0.80143** on Kaggle.

It tests three ideas against that result:
1. **XGBoost** on the v2 features.
2. **Group features**: fare per person, ticket-group size, sex × class, and a family-survival feature.
3. **Probability blending** of Random Forest, XGBoost and logistic regression, and a hybrid that applies the family-group rule on top.

**Leakage-safe evaluation.** Every number below comes from the same repeated stratified cross-validation (5 folds × 5 repeats = 25 splits). Anything learned from survival labels (age medians, family survival rates, the family rules) is rebuilt **inside each training fold**, so a validation passenger's own outcome is never used to predict them.

Kaggle scores are only reported for files that were actually submitted.

In [1]:
import glob, os, warnings
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
warnings.filterwarnings('ignore')

def find_file(name):
    # On Kaggle (/kaggle/input/...) or locally (the repo's data/ folder)
    for pattern in [f'/kaggle/input/**/{name}', f'../data/{name}', f'data/{name}']:
        hits = glob.glob(pattern, recursive=True)
        if hits:
            return hits[0]
    raise FileNotFoundError(f"{name} not found. On Kaggle: '+ Add Input' -> Titanic competition. "
                            f"Locally: download it from https://www.kaggle.com/competitions/titanic/data into data/.")

train_data = pd.read_csv(find_file('train.csv'))
test_data = pd.read_csv(find_file('test.csv'))
print(train_data.shape, test_data.shape)

(891, 12) (418, 11)


## Features
Label-free features are computed once on train and test together, because they never use `Survived`. Ticket counts include test passengers, which is fine because only their ticket numbers are used, not outcomes.

In [2]:
full = pd.concat([train_data, test_data], ignore_index=True)

full['Title'] = full['Name'].str.extract(r',\s*([^\.]+)\.')[0]
full['Title'] = full['Title'].replace(['Mlle', 'Ms'], 'Miss').replace('Mme', 'Mrs')
full.loc[~full['Title'].isin(['Mr', 'Mrs', 'Miss', 'Master']), 'Title'] = 'Rare'
full['Surname'] = full['Name'].str.split(',').str[0]
full['FamilySize'] = full['SibSp'] + full['Parch'] + 1
full['HasCabin'] = full['Cabin'].notna().astype(int)
full['Embarked'] = full['Embarked'].fillna('S')
full['Fare'] = full['Fare'].fillna(full.loc[full['Pclass'] == 3, 'Fare'].median())

# Group features
full['TicketGroupSize'] = full.groupby('Ticket')['Ticket'].transform('count')
full['FarePerPerson'] = full['Fare'] / full['TicketGroupSize']
full['SexPclass'] = full['Sex'] + '_' + full['Pclass'].astype(str)
full['IsBoy'] = full['Title'] == 'Master'
full['IsWoman'] = full['Sex'] == 'female'
full['WomanOrChild'] = full['IsBoy'] | full['IsWoman']
full['Group'] = (full['Surname'] + '-' + full['Pclass'].astype(str) + '-' + full['Ticket'].str[:-1] + '-'
                 + full['Embarked'] + '-' + full['Fare'].round(2).astype(str))

n_train = len(train_data)
train_full = full.iloc[:n_train].copy()
train_full['Survived'] = train_data['Survived'].values
test_full = full.iloc[n_train:].copy()
y = train_full['Survived'].values

full[['TicketGroupSize', 'FarePerPerson']].describe().round(2)

,TicketGroupSize,FarePerPerson
count,1309.00,1309.00
mean,2.10,14.75
std,1.78,13.55
min,1.00,0.00
25%,1.00,7.55
50%,1.00,8.05
75%,3.00,15.00
max,11.00,128.08


## Fold-safe features
These use survival labels, so they are computed from the **training fold only** and then applied to the validation fold (or, for the final model, from all of train and applied to test).
- **Age by title**: missing ages get the median age for the title in the training fold.
- **FamilySurvival**: the survival rate of the *other* women and boys in a passenger's family group (or ticket, if no family match) in the training fold. 0.5 means unknown.

In [3]:
def fold_features(train_df, apply_df, is_train_part):
    df = apply_df.copy()
    age_by_title = train_df.groupby('Title')['Age'].median()
    df['AgeT'] = df['Age'].fillna(df['Title'].map(age_by_title))

    wc = train_df[train_df['WomanOrChild']]
    fam = np.full(len(df), 0.5)
    for key in ['Ticket', 'Group']:          # Group (more specific) overrides Ticket
        sums = wc.groupby(key)['Survived'].sum()
        counts = wc.groupby(key)['Survived'].count()
        s = df[key].map(sums).fillna(0).values
        c = df[key].map(counts).fillna(0).values
        if is_train_part:                     # leave the passenger's own outcome out
            own = (df['WomanOrChild'] & df[key].isin(counts.index)).values
            s = s - np.where(own, df['Survived'].values, 0)
            c = c - own.astype(int)
        has = c > 0
        fam = np.where(has, s / np.maximum(c, 1), fam)
    df['FamilySurvival'] = fam
    return df

def design_matrix(df, columns):
    X = pd.get_dummies(df[columns], columns=[c for c in ['Sex', 'Embarked', 'Title', 'SexPclass'] if c in columns], dtype=float)
    return X

V2 = ['Pclass', 'Sex', 'AgeT', 'SibSp', 'Parch', 'Fare', 'Embarked', 'FamilySize', 'Title', 'HasCabin']
GROUP = V2 + ['FarePerPerson', 'TicketGroupSize', 'SexPclass']
GROUP_FAM = GROUP + ['FamilySurvival']
print(len(V2), len(GROUP), len(GROUP_FAM), 'features')

10 13 14 features


## Models and the family-group rule

In [4]:
def make_rf():  return RandomForestClassifier(n_estimators=300, max_depth=5, min_samples_leaf=2, random_state=42)
def make_xgb(): return XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.8,
                                     colsample_bytree=0.8, random_state=42, eval_metric='logloss', n_jobs=1)
def make_lr():  return LogisticRegression(max_iter=2000, C=0.5)

def fit_predict_proba(make_model, train_df, apply_df, columns, scale=False):
    Xtr = design_matrix(train_df, columns)
    Xap = design_matrix(apply_df, columns).reindex(columns=Xtr.columns, fill_value=0)
    if scale:
        sc = StandardScaler().fit(Xtr); Xtr, Xap = sc.transform(Xtr), sc.transform(Xap)
    model = make_model().fit(Xtr, train_df['Survived'].values)
    return model.predict_proba(Xap)[:, 1]

def family_rule_adjust(train_df, apply_df, base_preds):
    """Apply the woman-child-group overrides to any base predictions."""
    preds = np.array(base_preds).copy()
    wc = train_df[train_df['WomanOrChild']]
    by_group = wc.groupby('Group')['Survived'].mean()
    by_ticket = wc.groupby('Ticket')['Survived'].mean()
    for i, (_, r) in enumerate(apply_df.iterrows()):
        rate = by_group.get(r['Group'], by_ticket.get(r['Ticket'], None))
        if rate is None:
            continue
        if r['IsWoman'] and rate == 0: preds[i] = 0
        if r['IsBoy'] and rate == 1:   preds[i] = 1
    return preds

## Repeated cross-validation
All candidates are scored on the same 25 splits, so their scores are directly comparable.

In [5]:
def candidate_predictions(tr, va):
    trf = fold_features(tr, tr, True)
    vaf = fold_features(tr, va, False)
    out = {}
    out['Gender rule'] = vaf['IsWoman'].astype(int).values
    out['Family-group rule (current best)'] = family_rule_adjust(trf, vaf, vaf['IsWoman'].astype(int).values)
    p_rf  = fit_predict_proba(make_rf,  trf, vaf, V2)
    out['Random Forest, v2 features'] = (p_rf > 0.5).astype(int)
    p_xgb = fit_predict_proba(make_xgb, trf, vaf, V2)
    out['XGBoost, v2 features'] = (p_xgb > 0.5).astype(int)
    p_xgb_g = fit_predict_proba(make_xgb, trf, vaf, GROUP)
    out['XGBoost + group features'] = (p_xgb_g > 0.5).astype(int)
    p_xgb_f = fit_predict_proba(make_xgb, trf, vaf, GROUP_FAM)
    out['XGBoost + group + FamilySurvival'] = (p_xgb_f > 0.5).astype(int)
    p_rf_f = fit_predict_proba(make_rf,  trf, vaf, GROUP_FAM)
    p_lr_f = fit_predict_proba(make_lr,  trf, vaf, GROUP_FAM, scale=True)
    out['Random Forest + group + FamilySurvival'] = (p_rf_f > 0.5).astype(int)
    out['Logistic regression + group + FamilySurvival'] = (p_lr_f > 0.5).astype(int)
    blend = (p_rf_f + p_xgb_f + p_lr_f) / 3
    out['Blend (RF + XGB + LR probabilities)'] = (blend > 0.5).astype(int)
    out['Blend + family-group rule'] = family_rule_adjust(trf, vaf, (blend > 0.5).astype(int))
    return out

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)
scores = {}
for tr_idx, va_idx in cv.split(train_full, y):
    tr, va = train_full.iloc[tr_idx], train_full.iloc[va_idx]
    for name, pred in candidate_predictions(tr, va).items():
        scores.setdefault(name, []).append((pred == va['Survived'].values).mean())

cv_table = pd.DataFrame({
    'CV accuracy (%)': {k: np.mean(v) * 100 for k, v in scores.items()},
    'Std (%)': {k: np.std(v) * 100 for k, v in scores.items()},
}).round(2).sort_values('CV accuracy (%)', ascending=False)
cv_table

,CV accuracy (%),Std (%)
Blend + family-group rule,84.47,2.32
Blend (RF + XGB + LR probabilities),84.42,2.40
Random Forest + group + FamilySurvival,84.40,2.38
XGBoost + group + FamilySurvival,84.29,2.18
Family-group rule (current best),84.13,2.55
XGBoost + group features,83.91,1.97
Logistic regression + group + FamilySurvival,83.86,2.40
"XGBoost, v2 features",83.39,2.15
"Random Forest, v2 features",83.10,2.25
Gender rule,78.68,2.57


### Paired comparison against the current best
Because every candidate is scored on the same 25 splits, we can compare them split by split. "Wins" counts the splits where a candidate beats the family-group rule.

In [6]:
best = np.array(scores['Family-group rule (current best)'])
paired = pd.DataFrame({
    'Mean difference (pts)': {k: (np.array(v) - best).mean() * 100 for k, v in scores.items()},
    'Wins / ties / losses vs best': {k: f"{(np.array(v) > best).sum()} / {(np.array(v) == best).sum()} / {(np.array(v) < best).sum()}" for k, v in scores.items()},
}).round(2).sort_values('Mean difference (pts)', ascending=False)
paired

,Mean difference (pts),Wins / ties / losses vs best
Blend + family-group rule,0.34,13 / 2 / 10
Blend (RF + XGB + LR probabilities),0.29,13 / 1 / 11
Random Forest + group + FamilySurvival,0.27,13 / 6 / 6
XGBoost + group + FamilySurvival,0.16,12 / 2 / 11
Family-group rule (current best),0.00,0 / 25 / 0
XGBoost + group features,-0.22,9 / 4 / 12
Logistic regression + group + FamilySurvival,-0.27,8 / 4 / 13
"XGBoost, v2 features",-0.74,7 / 5 / 13
"Random Forest, v2 features",-1.03,5 / 3 / 17
Gender rule,-5.45,0 / 0 / 25


## Test-set predictions for the new candidates
These files are saved but **not yet submitted**, so no Kaggle score is claimed for them.

In [7]:
trf_all = fold_features(train_full, train_full, True)
tef_all = fold_features(train_full, test_full, False)

p_xgb   = fit_predict_proba(make_xgb, trf_all, tef_all, V2)
p_xgb_f = fit_predict_proba(make_xgb, trf_all, tef_all, GROUP_FAM)
p_rf_f  = fit_predict_proba(make_rf,  trf_all, tef_all, GROUP_FAM)
p_lr_f  = fit_predict_proba(make_lr,  trf_all, tef_all, GROUP_FAM, scale=True)
blend   = (p_rf_f + p_xgb_f + p_lr_f) / 3

family = family_rule_adjust(trf_all, tef_all, tef_all['IsWoman'].astype(int).values)
candidates = {
    'candidate_xgboost_v2.csv': (p_xgb > 0.5).astype(int),
    'candidate_xgboost_group_family.csv': (p_xgb_f > 0.5).astype(int),
    'candidate_blend.csv': (blend > 0.5).astype(int),
    'candidate_blend_plus_family_rule.csv': family_rule_adjust(trf_all, tef_all, (blend > 0.5).astype(int)),
}
rows = []
for fname, pred in candidates.items():
    pd.DataFrame({'PassengerId': test_data['PassengerId'], 'Survived': pred}).to_csv(fname, index=False)
    rows.append({'file': fname, 'predicted survivors': int(pred.sum()),
                 'differs from family-group submission': int((pred != family).sum())})
pd.DataFrame(rows)

,file,predicted survivors,differs from family-group submission
0,candidate_xgboost_v2.csv,152,41
1,candidate_xgboost_group_family.csv,151,32
2,candidate_blend.csv,151,18
3,candidate_blend_plus_family_rule.csv,151,18
